In [ ]:
import os
import copy
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from astropy.io import fits
from astropy.table import Table
from torch.utils.data import TensorDataset, DataLoader, random_split

warnings.filterwarnings("ignore")
BANDS = ["J", "H", "K", "G", "BP", "RP"]


class Config:
    MODE = "predict"   # "train" or "predict"
    CKPT_PATH = "./ MagDist-MLP.pth"

    TRAIN_DATA_PATH = r"gm"
    PREDICT_IN_FITS = r"predicted_params"
    PREDICT_OUT_FITS = r"prediction.fits"

    PRED_COL_FEH = "[M/H]_2"
    PRED_COL_EBV = "ebv"

    COL_EBV = "ebv_1"
    COL_PLX = "parallax"
    COL_TEFF = "TEFF"
    COL_LOGG = "LOGG"
    COL_FEH = "M_H"

    TRAIN_MAG_COLS = {
        "J": "Jmag", "H": "Hmag", "K": "Kmag",
        "G": "Gmag", "BP": "BPmag", "RP": "RPmag"
    }
    PRED_MAG_COLS = {
        "J": "Jmag", "H": "Hmag", "K": "Kmag",
        "G": "phot_g_mean_mag", "BP": "phot_bp_mean_mag", "RP": "phot_rp_mean_mag"
    }

    R_LAMBDA = {"J": 0.650, "H": 0.327, "K": 0.161, "G": 2.497, "BP": 3.209, "RP": 1.888}

    EPOCHS = 100
    BATCH_SIZE = 256
    LR = 1e-3
    HIDDEN = 64
    VAL_RATIO = 0.2
    ALPHA = 1.0
    BETA = 4.0
    SEED = 42
    USE_GPU = True


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_fits_path(path):
    if os.path.isfile(path):
        return path
    if os.path.isdir(path):
        files = [os.path.join(path, x) for x in os.listdir(path) if x.lower().endswith((".fits", ".fit", ".fits.gz", ".fit.gz"))]
        if files:
            return sorted(files)[0]
    raise FileNotFoundError(path)


def fit_scaler(x):
    med = np.nanmedian(x, axis=0)
    q1 = np.nanpercentile(x, 25, axis=0)
    q3 = np.nanpercentile(x, 75, axis=0)
    iqr = q3 - q1
    iqr[iqr == 0] = 1e-6
    return {"median": med.astype(np.float32), "iqr": iqr.astype(np.float32)}


def transform_scaler(x, scaler):
    return ((x - scaler["median"]) / scaler["iqr"]).astype(np.float32)


def load_train_data(cfg):
    path = resolve_fits_path(cfg.TRAIN_DATA_PATH)
    with fits.open(path) as hdul:
        tab = hdul[1].data

    ebv = tab[cfg.COL_EBV]
    plx = tab[cfg.COL_PLX]
    teff = tab[cfg.COL_TEFF]
    logg = tab[cfg.COL_LOGG]
    feh = tab[cfg.COL_FEH]
    mags = {b: tab[cfg.TRAIN_MAG_COLS[b]] for b in BANDS}

    mask = (plx > 0) & np.isfinite(plx) & np.isfinite(ebv) & np.isfinite(teff) & np.isfinite(logg) & np.isfinite(feh)
    for b in BANDS:
        mask &= np.isfinite(mags[b])

    ebv = np.asarray(ebv[mask], dtype=np.float32)
    plx = np.asarray(plx[mask], dtype=np.float32)
    teff = np.asarray(teff[mask], dtype=np.float32)
    logg = np.asarray(logg[mask], dtype=np.float32)
    feh = np.asarray(feh[mask], dtype=np.float32)
    mags = {b: np.asarray(mags[b][mask], dtype=np.float32) for b in BANDS}

    mu = 10.0 - 5.0 * np.log10(plx)
    abs_mags = []
    app_mags = []
    for b in BANDS:
        A = cfg.R_LAMBDA[b] * ebv
        m = mags[b]
        M = m - mu - A
        abs_mags.append(M)
        app_mags.append(m)

    y_M = np.stack(abs_mags, axis=1).astype(np.float32)
    y_mu = mu.reshape(-1, 1).astype(np.float32)
    x = np.stack([teff, logg, feh, ebv] + app_mags, axis=1).astype(np.float32)
    return x, y_M, y_mu


def load_predict_data(cfg):
    path = resolve_fits_path(cfg.PREDICT_IN_FITS)
    with fits.open(path) as hdul:
        data = hdul[1].data
        table = Table(data)

    teff = np.asarray(table[cfg.COL_TEFF], dtype=np.float32)
    logg = np.asarray(table[cfg.COL_LOGG], dtype=np.float32)
    feh = np.asarray(table[cfg.PRED_COL_FEH], dtype=np.float32)
    ebv = np.asarray(table[cfg.PRED_COL_EBV], dtype=np.float32)

    mag_cols = cfg.PRED_MAG_COLS
    app_mags = [np.asarray(table[mag_cols[b]], dtype=np.float32) for b in BANDS]
    x = np.stack([teff, logg, feh, ebv] + app_mags, axis=1).astype(np.float32)
    return table, x


class MinimalMLP(nn.Module):
    def __init__(self, in_dim=10, hidden=64, out_m=6):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.head_M = nn.Linear(hidden, out_m)
        self.head_mu = nn.Linear(hidden, 1)

    def forward(self, x):
        z = self.backbone(x)
        return self.head_M(z), self.head_mu(z)


def train_model(cfg):
    x, y_M, y_mu = load_train_data(cfg)
    scaler = fit_scaler(x)
    x = transform_scaler(x, scaler)

    ds = TensorDataset(torch.tensor(x), torch.tensor(y_M), torch.tensor(y_mu))
    n_val = int(len(ds) * cfg.VAL_RATIO)
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(cfg.SEED))

    train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)

    device = torch.device("cuda" if (cfg.USE_GPU and torch.cuda.is_available()) else "cpu")
    model = MinimalMLP(in_dim=x.shape[1], hidden=cfg.HIDDEN).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.LR)
    mse = nn.MSELoss()

    best_loss = np.inf
    best_state = None

    for epoch in range(cfg.EPOCHS):
        model.train()
        for xb, yMb, ymub in train_loader:
            xb, yMb, ymub = xb.to(device), yMb.to(device), ymub.to(device)
            pred_M, pred_mu = model(xb)
            loss = cfg.ALPHA * mse(pred_M, yMb) + cfg.BETA * mse(pred_mu, ymub)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        n = 0
        with torch.no_grad():
            for xb, yMb, ymub in val_loader:
                xb, yMb, ymub = xb.to(device), yMb.to(device), ymub.to(device)
                pred_M, pred_mu = model(xb)
                loss = cfg.ALPHA * mse(pred_M, yMb) + cfg.BETA * mse(pred_mu, ymub)
                val_loss += loss.item() * len(xb)
                n += len(xb)
        val_loss /= max(n, 1)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:03d} | val_loss = {val_loss:.6f}")

    model.load_state_dict(best_state)
    torch.save({"model_state": model.state_dict(), "scaler": scaler}, cfg.CKPT_PATH)
    print("Saved:", cfg.CKPT_PATH)


def predict(cfg):
    ckpt = torch.load(cfg.CKPT_PATH, map_location="cpu")
    scaler = ckpt["scaler"]

    table, x = load_predict_data(cfg)
    x = transform_scaler(x, scaler)

    device = torch.device("cuda" if (cfg.USE_GPU and torch.cuda.is_available()) else "cpu")
    model = MinimalMLP(in_dim=x.shape[1], hidden=cfg.HIDDEN).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    with torch.no_grad():
        xt = torch.tensor(x, dtype=torch.float32).to(device)
        pred_M, pred_mu = model(xt)
        pred_M = pred_M.cpu().numpy()
        pred_mu = pred_mu.cpu().numpy().ravel()

    for i, b in enumerate(BANDS):
        table[f"M_pred_{b}"] = pred_M[:, i]
    table["mu_pred"] = pred_mu
    table["d_pred_kpc"] = 10 ** ((pred_mu + 5.0) / 5.0) / 1000.0
    table.write(cfg.PREDICT_OUT_FITS, overwrite=True)
    print("Saved:", cfg.PREDICT_OUT_FITS)


def main():
    cfg = Config()
    set_seed(cfg.SEED)
    if cfg.MODE == "train":
        train_model(cfg)
    elif cfg.MODE == "predict":
        predict(cfg)
    else:
        raise ValueError("MODE must be 'train' or 'predict'")


if __name__ == "__main__":
    main()